In [1]:
# Finite difference vs RK4 shooting for f''(0). Written by Claude Code (claude-opus-5-5), 2026-10-25; see AI-USE.md § 2
# Limit: this compares the two methods' f''(0) only, not the velocity profile g = u/U.
import timeit

import numpy as np
import pandas as pd

import sys
from pathlib import Path

# The repository root, found from any working directory, so that the src package imports
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "friction").is_dir())
sys.path.insert(0, str(ROOT))
from src.friction.blasius_shooting.shooting_blasius import (F, secant_for_shooting, GUESS_ddf, GUESS2_ddf,
                                                            STEP_SIZE, ETA_ARRAY)
from src.friction.blasius_shooting.fd_blasius import blasius_fd, N, h, eta_max, PICARD_ITERATIONS

pd.set_option("display.precision", 12)

# Exact f''(0) for f''' + 1/2 f f'' = 0: sigma/sqrt(2), White and Majdalani (2022), p.248, footnote to Eq. (4-64)
EXACT_ddf = 0.3320573362151963

# Same step and eta_max as the RK4 shooting
assert np.isclose(h, STEP_SIZE, rtol=1e-12, atol=0) and eta_max == ETA_ARRAY[-1]

In [2]:
# Accuracy of f''(0), both methods on the same grid
# Limit: eta_max is about 9.970 for both methods, not infinity.
g, f, ddf_fd = blasius_fd()
s_star, num_iter, s_history, f_history = secant_for_shooting(F, GUESS_ddf, GUESS2_ddf)

# Max-norm error (MMA3001 notes 10.1); for the single value f''(0) it is the absolute error
accuracy = pd.DataFrame({"f''(0)": [ddf_fd[-1], s_star],
                         "max-norm error": [np.abs(ddf_fd[-1] - EXACT_ddf), np.abs(s_star - EXACT_ddf)]},
                        index=["central difference", "RK4 shooting"])

# Picard convergence: the change in f''(0) from one iteration to the next
# Limit: Picard convergence is checked for this case, not proved in general.
change = np.abs(np.diff(ddf_fd))
print(f"Picard change in f''(0), iteration {PICARD_ITERATIONS - 1} -> {PICARD_ITERATIONS}: {change[-1]:.1e}; "
      f"first below 1e-6 at iteration {np.argmax(change < 1e-6) + 2}")
assert change[-1] < 1e-8 and np.abs(ddf_fd[-1] - EXACT_ddf) < 1e-3  # the 1/2 stencil gives an error of 0.14
accuracy

Picard change in f''(0), iteration 19 -> 20: 8.1e-11; first below 1e-6 at iteration 13


,f''(0),max-norm error
central difference,0.332074725114,0.000017388898
RK4 shooting,0.332057350798,0.000000014583


In [3]:
# Cost: median wall time of one full solve over timeit repeats
NUMBER, REPEATS = 10, 7
fd_time = np.median(timeit.repeat(blasius_fd, number=NUMBER, repeat=REPEATS))/NUMBER
shooting_time = np.median(timeit.repeat(lambda: secant_for_shooting(F, GUESS_ddf, GUESS2_ddf),
                                        number=NUMBER, repeat=REPEATS))/NUMBER

# Machine-independent work beside the times
cost = pd.DataFrame({"median time per solve [ms]": [fd_time*1e3, shooting_time*1e3],
                     "work": [f"{PICARD_ITERATIONS} Thomas solves of {N - 1} unknowns",
                              f"{len(s_history)} RK4 integrations of {len(ETA_ARRAY) - 1} steps, 4 RHS calls each"]},
                    index=["central difference", "RK4 shooting"])

# Limit: the times are of these pure-Python loops on this machine, not of the methods in general.
# Limit: the finite difference runs all 20 Picard iterations while shooting stops at |F| < 1e-6, so the
# two are not compared at equal accuracy.
# Limit: one h only, so the finite difference's second-order convergence is not shown here.
cost

,median time per solve [ms],work
central difference,2.3147468,20 Thomas solves of 140 unknowns
RK4 shooting,10.2100550,"9 RK4 integrations of 141 steps, 4 RHS calls each"
